[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/bayesian/mcmc_poisson_mixture.ipynb)

# MCMC for Mixture Models: Inferring Earthquake Regimes

Build a Metropolis-Hastings sampler from scratch to infer hidden activity regimes in earthquake count data.
We fit a two-component Poisson mixture model, getting full posterior distributions for each parameter.

**Blog post:** [sesen.ai/blog/mcmc-poisson-mixture-earthquake-regimes](https://sesen.ai/blog/mcmc-poisson-mixture-earthquake-regimes)

In [ ]:
import numpy as np
from scipy.stats import poisson, gaussian_kde
import matplotlib.pyplot as plt

np.random.seed(42)

## The Data: 107 Years of Earthquakes

Number of major earthquakes (Richter > 7) worldwide per year, 1900–2006.
Source: Zucchini, MacDonald & Langrock (2016), *Hidden Markov Models for Time Series*, p. 10.

In [ ]:
eq = np.array([
    13, 14,  8, 10, 16, 26, 32, 27, 18, 32, 36, 24, 22, 23, 22, 18, 25,
    21, 21, 14,  8, 11, 14, 23, 18, 17, 19, 20, 22, 19, 13, 26, 13, 14,
    22, 24, 21, 22, 26, 21, 23, 24, 27, 41, 31, 27, 35, 26, 28, 36, 39,
    21, 17, 22, 17, 19, 15, 34, 10, 15, 22, 18, 15, 20, 15, 22, 19, 16,
    30, 27, 29, 23, 20, 16, 21, 21, 25, 16, 18, 15, 18, 14, 10, 15,  8,
    15,  6, 11,  8,  7, 18, 16, 13, 12, 13, 20, 15, 16, 12, 18, 15, 16,
    13, 15, 16, 11, 11,
])
print(f'{len(eq)} years, mean={eq.mean():.1f}, min={eq.min()}, max={eq.max()}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(eq, bins=np.arange(eq.min() - 0.5, eq.max() + 1.5, 1),
        color='#4C72B0', edgecolor='white', alpha=0.85)
ax.set_xlabel('Number of major earthquakes per year')
ax.set_ylabel('Frequency')
ax.set_title('Major Earthquakes per Year (Richter > 7), 1900\u20132006')
ax.axvline(np.mean(eq), color='red', linestyle='--', linewidth=1.5,
           label=f'Mean = {np.mean(eq):.1f}')
ax.legend()
plt.tight_layout()
plt.show()

## The Model: Two-Component Poisson Mixture

Each year's earthquake count is drawn from one of two Poisson distributions:

$$P(x_i) = \delta_1 \cdot \text{Pois}(x_i \mid \lambda_1) + \delta_2 \cdot \text{Pois}(x_i \mid \lambda_2)$$

Three free parameters: $\lambda_1$ (quiet-regime rate), $\lambda_2$ (active-regime rate), $\delta_1$ (quiet-regime probability).

In [ ]:
def poisson_mix_loglik(data, lam1, lam2, d1):
    """Log-likelihood of a 2-component Poisson mixture."""
    d2 = 1.0 - d1
    mix = d1 * poisson.pmf(data, lam1) + d2 * poisson.pmf(data, lam2)
    mix = np.maximum(mix, 1e-300)  # protect against log(0)
    return np.sum(np.log(mix))

## Metropolis-Hastings Sampler

We propose new parameters from a multivariate normal centred on the current position,
then accept or reject based on the likelihood ratio.

In [ ]:
def run_mcmc(data, n_iter=1000, burn_in=100):
    # Proposal covariance (from the original R code)
    sigma = np.diag([1.0, 1.0, 0.01])
    # Storage: [lam1, lam2, d1, d2, loglik]
    params = np.full((n_iter + 1, 5), np.nan)

    # Initial values
    params[0] = [10.0, 20.0, 0.3, 0.7,
                 poisson_mix_loglik(data, 10.0, 20.0, 0.3)]

    n_accept = 0
    for i in range(n_iter):
        cur_ll = params[i, 4]

        # Propose from multivariate normal centred on current position
        current = params[i, :3]
        prop = np.random.multivariate_normal(mean=current, cov=sigma)
        p_lam1, p_lam2, p_d1 = prop

        # Enforce constraints
        if p_lam1 <= 0 or p_lam2 <= 0 or p_d1 <= 0 or p_d1 >= 1:
            prop_ll = -np.inf
        else:
            prop_ll = poisson_mix_loglik(data, p_lam1, p_lam2, p_d1)

        # Accept/reject
        log_ratio = prop_ll - cur_ll
        if np.log(np.random.uniform()) < min(0.0, log_ratio):
            params[i + 1] = [p_lam1, p_lam2, p_d1, 1 - p_d1, prop_ll]
            n_accept += 1
        else:
            params[i + 1] = params[i]

    posterior = params[burn_in + 1:]
    return params, posterior, n_accept / n_iter

In [ ]:
all_params, posterior, accept_rate = run_mcmc(eq, n_iter=1000, burn_in=100)

print(f'Acceptance rate: {accept_rate:.1%}')
print(f'lambda1: {posterior[:, 0].mean():.1f} \u00b1 {posterior[:, 0].std():.1f}')
print(f'lambda2: {posterior[:, 1].mean():.1f} \u00b1 {posterior[:, 1].std():.1f}')
print(f'delta1:  {posterior[:, 2].mean():.2f} \u00b1 {posterior[:, 2].std():.2f}')
print(f'\n95% credible intervals:')
for name, col in [('lambda1', 0), ('lambda2', 1), ('delta1', 2)]:
    lo, hi = np.percentile(posterior[:, col], [2.5, 97.5])
    print(f'  {name}: ({lo:.2f}, {hi:.2f})')

## Trace Plots

Trace plots show how each parameter evolves over MCMC iterations.
The grey region is burn-in (first 100 iterations), discarded from the posterior.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
labels = [r'$\lambda_1$', r'$\lambda_2$', r'$\delta_1$']
cols = [0, 1, 2]
colours = ['#4C72B0', '#DD8452', '#55A868']
burn_in = 100

for ax, label, col, colour in zip(axes, labels, cols, colours):
    ax.plot(all_params[:, col], linewidth=0.5, color=colour, alpha=0.8)
    ax.axvspan(0, burn_in, color='grey', alpha=0.15, label='Burn-in')
    ax.axvline(burn_in, color='grey', linestyle='--', linewidth=0.8)
    ax.set_ylabel(label)
    pm = np.mean(all_params[burn_in + 1:, col])
    ax.axhline(pm, color='black', linestyle=':', linewidth=1,
               label=f'Post. mean = {pm:.2f}')
    ax.legend(loc='upper right', fontsize=9)

axes[-1].set_xlabel('Iteration')
axes[0].set_title('MCMC Trace Plots')
plt.tight_layout()
plt.show()

## Marginal Posterior Distributions

Unlike EM (which gives point estimates), MCMC gives full posterior distributions with uncertainty.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
param_names = [r'$\lambda_1$', r'$\lambda_2$', r'$\delta_1$']

for ax, label, col, colour in zip(axes, param_names, [0, 1, 2], colours):
    samples = posterior[:, col]
    ax.hist(samples, bins=35, color=colour, alpha=0.6, edgecolor='white',
            density=True, label='Histogram')
    if np.std(samples) > 1e-8:
        kde = gaussian_kde(samples)
        xs = np.linspace(samples.min(), samples.max(), 300)
        ax.plot(xs, kde(xs), color='black', linewidth=1.5, label='KDE')
    ax.axvline(np.mean(samples), color='red', linestyle='--', linewidth=1.2,
               label=f'Mean = {np.mean(samples):.2f}')
    ax.set_xlabel(label)
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)

fig.suptitle('Marginal Posterior Distributions', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## Posterior Pairs Plot

The pairs plot shows correlations between parameters in the posterior.

In [ ]:
param_cols = [0, 1, 2]
n_params = len(param_cols)

fig, axes = plt.subplots(n_params, n_params, figsize=(9, 9))

for row in range(n_params):
    for ci in range(n_params):
        ax = axes[row, ci]
        x_data = posterior[:, param_cols[ci]]
        y_data = posterior[:, param_cols[row]]

        if ci > row:
            ax.set_visible(False)
            continue

        if row == ci:
            ax.hist(x_data, bins=30, color=colours[row], alpha=0.7,
                    edgecolor='white', density=True)
            if np.std(x_data) > 1e-8:
                kde = gaussian_kde(x_data)
                xs = np.linspace(x_data.min(), x_data.max(), 200)
                ax.plot(xs, kde(xs), color='black', linewidth=1.2)
        else:
            ax.scatter(x_data, y_data, s=4, alpha=0.3, color='#4C72B0')

        if row == n_params - 1:
            ax.set_xlabel(param_names[ci])
        else:
            ax.set_xticklabels([])
        if ci == 0:
            ax.set_ylabel(param_names[row])
        else:
            ax.set_yticklabels([])

fig.suptitle('Posterior Pairs Plot (post burn-in)', y=1.01, fontsize=14)
plt.tight_layout()
plt.show()

## Fitted Mixture Overlay

Using the posterior means, we overlay the fitted two-component Poisson mixture on the data.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

x_range = np.arange(0, eq.max() + 5)
lam1_fit = posterior[:, 0].mean()
lam2_fit = posterior[:, 1].mean()
d1_fit = posterior[:, 2].mean()
d2_fit = 1.0 - d1_fit

ax.hist(eq, bins=np.arange(eq.min() - 0.5, eq.max() + 1.5, 1),
        density=True, color='#cccccc', edgecolor='white', alpha=0.9,
        label='Empirical histogram')

mix_pmf = d1_fit * poisson.pmf(x_range, lam1_fit) + d2_fit * poisson.pmf(x_range, lam2_fit)
comp1 = d1_fit * poisson.pmf(x_range, lam1_fit)
comp2 = d2_fit * poisson.pmf(x_range, lam2_fit)

ax.plot(x_range, mix_pmf, 'k-o', markersize=4, linewidth=2, label='Mixture (fitted)')
ax.plot(x_range, comp1, '--', color='#4C72B0', linewidth=1.5,
        label=f'Component 1: $\\lambda_1$={lam1_fit:.1f}, $\\delta_1$={d1_fit:.2f}')
ax.plot(x_range, comp2, '--', color='#DD8452', linewidth=1.5,
        label=f'Component 2: $\\lambda_2$={lam2_fit:.1f}, $\\delta_2$={d2_fit:.2f}')

ax.set_xlabel('Number of major earthquakes per year')
ax.set_ylabel('Probability')
ax.set_title('Fitted 2-Component Poisson Mixture (Posterior Means)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Exercises

1. **Different initial values.** Try starting from `lam1=25, lam2=10, d1=0.7` (swapped rates). Does the chain converge to the same posterior? What does this tell you about label switching?

2. **Longer chains.** Increase `n_iter` to 5,000 or 10,000. How much smoother do the marginal posteriors become?

3. **Three-component mixture.** Extend the model to 3 Poisson components. Does the data support a third regime, or does one component collapse to near-zero weight?

4. **Adaptive proposal.** Replace the diagonal proposal covariance with one estimated from the running chain (e.g., update $\Sigma$ every 200 iterations using the sample covariance of accepted samples). Does the acceptance rate improve?

5. **PyMC comparison.** Implement the same 2-component Poisson mixture in PyMC using `pm.Mixture` and `pm.Poisson`. Compare the posterior summaries to our hand-rolled MCMC.